In [19]:
import numpy as np

machines = []
buttons = []
buttons_np = []
joltages = []

def converter(s: str) -> str:
    return ''.join(['1' if c=='#' else '0' for c in s])

def converter2(s: str) -> np.ndarray:
    s = '[' + s[1:-1] + ']'
    return np.array(eval(s))

with open('input.txt', 'r') as file:
    for line in file.readlines():
        blocks = line[:-1].split(' ')

        machine = converter(blocks[0][1:-1])
        machines.append(machine)

        jolts = converter2(blocks[-1])
        joltages.append(jolts)
        n = len(jolts)
        
        button_block = []
        button_block_np = []
        for block in blocks[1:-1]:
            block = eval(block)
            if isinstance(block, int):
                block = (block, )

            block_np = np.zeros(n, dtype=int)
            for b in block:
                block_np[b] = 1

            button_block.append(block)
            button_block_np.append(block_np)
        
        buttons.append(button_block)
        buttons_np.append(button_block_np)


In [16]:
import heapq

class Node(object):
    def __init__(self, val: str, goal: str, steps: int):
        self.val = val
        self.goal = goal
        self.steps = steps

    def __lt__(self, other: 'Node') -> bool:
        """Sort by A* f(n) = g(n)"""
        return self.steps <= other.steps
    
def flip(bin_str: str, button: tuple[int]) -> str:
    s = list(bin_str)
    for i in button:
        s[i] = '1' if s[i] == '0' else '0'
    return ''.join(s)

def fewest_switches(root: str, buttons: list[tuple[int]], goal: str) -> int:
    visited = set()
    pqueue = [Node(root, goal, 0)]

    while pqueue:
        node = heapq.heappop(pqueue)

        if node.val == goal:
            return node.steps
        
        visited.add(node.val)

        for button in buttons:
            new_str = flip(node.val, button)
            if new_str not in visited:
                heapq.heappush(pqueue, Node(new_str, goal, node.steps+1))
            
    raise ValueError('Problem is solvable')

tot = 0
for goal, button_lst in zip(machines, buttons):
    n = len(goal)
    tot += fewest_switches('0'*n, button_lst, goal)

print(tot)
        

488


In [38]:
# treat as a linear algebra problem
# y = Bx, with min(Σ_i x_i) as constraint

import pulp
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, value, PULP_CBC_CMD

def solve_min_sum_x_integer(B_matrix, y_vector) -> int:
    """
    Solves Bx = y for integer x, minimizing the sum of coefficients in x (sum(x_i)).
    Assumes x_i >= 0 and are integers.
    """
    B = np.array(B_matrix, dtype=float)
    y = np.array(y_vector, dtype=float)
    num_rows, num_cols = B.shape
    
    # instantiate problem with objective function
    prob = LpProblem("Minimize_X_Sum", LpMinimize)
    
    # variational params specified as integers >= 0
    x_vars = [LpVariable(f"x_{i}", lowBound=0, cat='Integer') for i in range(num_cols)]
    
    # define objective function
    prob += lpSum(x_vars), "Sum_of_X_Coefficients"
    
    # define constraints
    for i in range(num_rows):
        row_sum = lpSum([B[i][j] * x_vars[j] for j in range(num_cols)])
        prob += row_sum == y[i], f"Constraint_Row_{i}"

    solver = PULP_CBC_CMD(msg=False)
    prob.solve(solver)

    if prob.status == pulp.LpStatusOptimal:
        optimal_x_values = [value(var) for var in x_vars]
        return np.array(optimal_x_values, dtype=int).sum().item()
    
    return -1

tot = 0
for y, bs in zip(joltages, buttons_np):
    B = np.stack(bs, axis=0).T
    y = y.T

    solve = solve_min_sum_x_integer(B, y)
    if solve == -1:
        raise ValueError('Solution not found')
    
    tot += solve
    


print(tot)




18771
